In [1]:
from fakt_ai.crewai_implementation import (
    semantic_scholar_crew, fakt_ai_crew, paper_analysis_crew, final_answer_crew, SemanticScholarTool
)
from fakt_ai.utils import _get_llm

from fakt_ai.langchain_implementation import semantic_scholar_search_chain, paper_analysis_chain

from IPython.display import Markdown, display
from semanticscholar import SemanticScholar
from dotenv import find_dotenv, load_dotenv
import os
from pprint import pprint
import json
import ast

from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate

load_dotenv()

True

In [2]:
llm =_get_llm("anthropic")
llm.invoke("hello whats your name")

AIMessage(content="I'm Claude, an AI assistant created by Anthropic. How can I help you today?", additional_kwargs={}, response_metadata={'id': 'msg_01GYwDJFxJhKGZ2ukc71vRvL', 'model': 'claude-3-5-sonnet-20241022', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'input_tokens': 12, 'output_tokens': 23}}, id='run-1fa2581a-1e02-4d83-b126-a262c2a95a06-0', usage_metadata={'input_tokens': 12, 'output_tokens': 23, 'total_tokens': 35, 'input_token_details': {}})

In [2]:
query = "Is flouride in toothpaste unhealthy?"
search_chain = semantic_scholar_search_chain()
# papers = search_chain.invoke({"query": query})

# len(papers)

In [3]:
search_chain.invoke({"query": "how many rs in strawberry"})

NotFoundError: Not Found

In [2]:
query = "Do vaccines cause autism?"
search_chain = semantic_scholar_search_chain()
papers = search_chain.invoke({"query": query})

In [9]:
pprint(dict(papers[3]))

{'abstract': 'Since vaccinologists are themselves ignorant of vaccine '
             'mechanisms, how can we expect epidemiologists to understand the '
             'mechanisms? So most epidemiological studies ignore mechanism of '
             'adverse event causation. If you ignore mechanism, you cannot '
             'design the study with appropriate controls. So the results of '
             'such epidemiological studies have to be discarded due to '
             'confounding.\n'
             '\n'
             "Proven mechanism\xa0involved in cow's milk contaminated vaccines "
             'causing folate receptor antibody related autism is described.',
 'authors': [{'authorId': '81879080', 'name': 'Vinu Arumugham'}],
 'citationCount': 2,
 'influentialCitationCount': 0,
 'journal': {'name': '', 'volume': ''},
 'paperId': '4f4e4038ff0b1c19ed104562af13e02d4a91f2b7',
 'publicationDate': None,
 'referenceCount': 37,
 'title': 'Epidemiological Studies That Ignore Mechanism Of Disease C

20

In [14]:
@tool
def semantic_scholar_search(query: str):
    """A tool that can be used to call the Semantic Scholar API and obtain information about academic papers."""
    try:
        ss = SemanticScholar(retry=True, api_key=os.getenv("SEMANTIC_SCHOLAR_API_KEY"))
        fields = [
            "paperId",
            "tldr",
            "abstract",
            "url",
            "title",
            "publicationDate",
            "journal",
            "referenceCount",
            "citationCount",
            "influentialCitationCount",
            "authors",
        ]
        results = ss.search_paper(query, fields=fields, limit=20)
        return results.items
    except Exception as e:
        return f"Failed to search Semantic Scholar for query '{query}': {e}"

prompt = PromptTemplate.from_template(
    "Search on semantic scholar for papers that support this query: {query}. "
    # "\nNote that there may not be papers supporting a given query. In which case, "
    # "it is ok and informative to return no papers."
)

llm = _get_llm("groq")
tools = [semantic_scholar_search]
llm_with_tool = llm.bind_tools(tools, tool_choice=True)

chain = prompt | llm

query = "What treatments were available to treat covid before the vaccines came out?"
o = llm_with_tool.invoke(
    # "Search on semantic scholar for papers that support this query: "
    "What treatments were available to treat covid before the vaccines came out?"
)

chain = llm_with_tool | (lambda x: x.tool_calls[0]["args"]) | semantic_scholar_search
output = chain.invoke("What treatments were available to treat covid before the vaccines came out?")

In [17]:
output[0]

{'paperId': '1a27bf9d67d284be3f9cb32fa1a304b49a6eebdc', 'url': 'https://www.semanticscholar.org/paper/1a27bf9d67d284be3f9cb32fa1a304b49a6eebdc', 'title': 'Observational Study on Antibody Response to COVID-19 Vaccines in PAtients with Gastro-Entero-PanCreatic Cancers and NeuroendocrIne NeoplAsms on Systemic TreatmEnts (VACCINATE)', 'abstract': 'The coronavirus disease-19 (COVID-19) pandemic dramatically impacted oncological patients’ care. Since the introduction of vaccines and the demonstration of their benefit on frail patients, COVID-19 vaccinations were indicated to also be beneficial to oncological population. However, data about the impact of anticancer-treatments and the timing between vaccinations and systemic therapy delivery were not available. We aimed to evaluate potential factors influencing the outcome of the COVID-19 vaccination in cancer patients. We prospectively collected data of patients undergoing the COVID-19 vaccination with gastro-entero-pancreatic and neuroendocr

In [12]:
o.tool_calls

[{'name': 'semantic_scholar_search',
  'args': {'query': 'COVID-19 treatments before vaccine'},
  'id': 'call_74he',
  'type': 'tool_call'}]

In [5]:
output = llm_with_tool.invoke("What treatments were available to treat covid before the vaccines came out?")

In [21]:
from langchain import hub
from langchain.agents import AgentExecutor, create_tool_calling_agent

# Get the prompt to use - can be replaced with any prompt that includes variables "agent_scratchpad" and "input"!
prompt = hub.pull("hwchase17/openai-tools-agent")
# Construct the tool calling agent
agent = create_tool_calling_agent(llm, tools, prompt)
# Create an agent executor by passing in the agent and tools
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor.invoke({
    "input": "What treatments were available to treat covid before the vaccines came out?"
})




> Entering new AgentExecutor chain...

Invoking: `brave_search` with `{'query': 'COVID-19 treatments before vaccines'}`


brave_search is not a valid tool, try one of [semantic_scholar_search].
Invoking: `semantic_scholar_search` with `{'query': 'COVID-19 treatments before vaccines'}`


[{'paperId': '1a27bf9d67d284be3f9cb32fa1a304b49a6eebdc', 'url': 'https://www.semanticscholar.org/paper/1a27bf9d67d284be3f9cb32fa1a304b49a6eebdc', 'title': 'Observational Study on Antibody Response to COVID-19 Vaccines in PAtients with Gastro-Entero-PanCreatic Cancers and NeuroendocrIne NeoplAsms on Systemic TreatmEnts (VACCINATE)', 'abstract': 'The coronavirus disease-19 (COVID-19) pandemic dramatically impacted oncological patients’ care. Since the introduction of vaccines and the demonstration of their benefit on frail patients, COVID-19 vaccinations were indicated to also be beneficial to oncological population. However, data about the impact of anticancer-treatments and the timing between vaccinat

{'input': 'What treatments were available to treat covid before the vaccines came out?',
 'output': "The available treatments for COVID-19 before the vaccines were available include:\n\n1. Antiviral drugs such as remdesivir, which was approved for emergency use by the FDA.\n2. Anti-inflammatory drugs such as dexamethasone, which was shown to reduce mortality in hospitalized patients.\n3. Monoclonal antibodies, which were approved for emergency use by the FDA.\n4. Convalescent plasma, which was also approved for emergency use by the FDA.\n5. Anticoagulants, which were used to prevent blood clots in hospitalized patients.\n\nIt's worth noting that these treatments were not always effective and had varying degrees of success in different patients. The development and distribution of vaccines have been a crucial step in controlling the spread of COVID-19 and reducing the number of hospitalizations and deaths."}

In [19]:
o.content

''

In [12]:
semantic_scholar_search.invoke({"query": "What treatments were available to treat covid before the vaccines came out?"})

[{'paperId': '6158e56a389063ca6c1bbbaab8c8eb22ceaf58eb', 'url': 'https://www.semanticscholar.org/paper/6158e56a389063ca6c1bbbaab8c8eb22ceaf58eb', 'title': 'Towards Broadening the COVID-19 Knowledge Base', 'abstract': 'When the coronavirus 2019 (COVID-19) pandemic struck during the first quarter of 2020, global attention trained in on the city of Wuhan, the capital of Hubei province in China, where the first cases were reported in December 2019. With its exponential spread to other countries over a short span of time, infecting millions of people and exacting a global death toll of over half a million during the first six months of its discovery,1 interest in the little-known yet devastating severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2) virus splurged. As quickly as the virus spread across the globe did the scientific publications on the subject go into circulation, a phenomenon unequalled compared to other pandemics, with an estimated rate of 181 articles published per d

In [2]:
crew = semantic_scholar_crew()
query = 'what medications were available to treat covid before the vaccines came out?'
output = crew.kickoff({'query': query})
papers: list[dict] = ast.literal_eval(output.raw)
papers

 [2024-10-13 16:32:25][DEBUG]: == Working Agent: Research Analyst
 [2024-10-13 16:32:25][INFO]: == Starting Task: Search on semantic scholar for papers that support this query: what medications were available to treat covid before the vaccines came out?. 
Note that there may not be papers supporting a given query. In which case, it is ok and informative to return no papers. 
 [2024-10-13 16:32:29][DEBUG]: == [Research Analyst] Task output: [{'paperId': '2936e5999c5d711b598a23bf875c7e041e8e88d4', 'url': 'https://www.semanticscholar.org/paper/2936e5999c5d711b598a23bf875c7e041e8e88d4', 'title': 'Impact of COVID-19 on Cardiovascular Patients and Review of Current COVID-19 Treatment Strategies', 'abstract': 'Coronavirus disease of 2019 ( COVID-19) has severely affected global health and caused a significant health burden worldwide, particularly in patients with cardiovascular insult Worldwide, the COVID-19 confirmed cases number reached 81,658,440 cases with 1,802,206 deaths stated to the w

[{'paperId': '2936e5999c5d711b598a23bf875c7e041e8e88d4',
  'url': 'https://www.semanticscholar.org/paper/2936e5999c5d711b598a23bf875c7e041e8e88d4',
  'title': 'Impact of COVID-19 on Cardiovascular Patients and Review of Current COVID-19 Treatment Strategies',
  'abstract': 'Coronavirus disease of 2019 ( COVID-19) has severely affected global health and caused a significant health burden worldwide, particularly in patients with cardiovascular insult Worldwide, the COVID-19 confirmed cases number reached 81,658,440 cases with 1,802,206 deaths stated to the world health organization by January 1, 2021 Many risk factors, such as the elderly, diabetes, chronic kidney disease (CKD), and cardiovascular illness, like coronary disease, cardiomyopathy, and hypertension;put people at high vulnerability with COVID-19 infection Many cardiovascular insults directly occur because of COVID-19 infection as myocarditis, pericarditis, heart failure, myocardial infarction, thromboembolic events, or arrhyt

In [3]:
analysis_crew = paper_analysis_crew()

paper_analyses = []
for paper in papers:
    paper_analysis = analysis_crew.kickoff({
        "query": query, 
        "paper": paper
    })
    paper_analyses.append(paper_analysis.raw)

2024-10-13 16:32:29,338 - 126579333955712 - __init__.py-__init__:538 - WARNING: Overriding of current TracerProvider is not allowed


 [2024-10-13 16:32:29][DEBUG]: == Working Agent: Paper Analyst
 [2024-10-13 16:32:29][INFO]: == Starting Task: We are trying to determine the truth or falisty of the following query
    
    what medications were available to treat covid before the vaccines came out?
    
    Here is a paper related to the query. Please write a summary of the contents and how the paper
    either supports or refutes the query.
    
    {'paperId': '2936e5999c5d711b598a23bf875c7e041e8e88d4', 'url': 'https://www.semanticscholar.org/paper/2936e5999c5d711b598a23bf875c7e041e8e88d4', 'title': 'Impact of COVID-19 on Cardiovascular Patients and Review of Current COVID-19 Treatment Strategies', 'abstract': 'Coronavirus disease of 2019 ( COVID-19) has severely affected global health and caused a significant health burden worldwide, particularly in patients with cardiovascular insult Worldwide, the COVID-19 confirmed cases number reached 81,658,440 cases with 1,802,206 deaths stated to the world health organizati

In [4]:
final_answer_crew = final_answer_crew()
final_answer = final_answer_crew.kickoff({
    "query": query,
    "paper_analyses": paper_analyses,
})
print(final_answer.raw)

2024-10-13 16:33:29,883 - 126579333955712 - __init__.py-__init__:538 - WARNING: Overriding of current TracerProvider is not allowed


 [2024-10-13 16:33:29][DEBUG]: == Working Agent: Final Answer Agent
 [2024-10-13 16:33:29][INFO]: == Starting Task: We are trying to determine the truth or falisty of the following query
    
    what medications were available to treat covid before the vaccines came out?
    
    We have already collected and analysed many scientific papers related to the query.
    Please summarise the scientific analyses and write a report determining the truth or falsity
    of the query.
    
    Paper analyses:
    ["The paper 'Impact of COVID-19 on Cardiovascular Patients and Review of Current COVID-19 Treatment Strategies' provides a comprehensive review of the impact of COVID-19 on cardiovascular patients and discusses current treatment strategies for COVID-19. The paper highlights the severity of the pandemic and its significant health burden worldwide, particularly in patients with cardiovascular comorbidities.\n\nThe paper mentions that the COVID-19 vaccine is the best promising plan agains

In [5]:
print(final_answer.raw)

The query about the medications available to treat COVID-19 before the vaccines came out can be confirmed based on the analysis of the provided papers. 

Several papers provide evidence that various medications were available to treat COVID-19 before the vaccines came out. For example, the paper "Different Treatment Approaches, Clinical Outcome, Effectiveness of the Drug towards Covid 19 with 5 Different Cases: A Case Series" provides evidence that certain medications, including dexamethasone, antimalarials (chloroquine/hydroxychloroquine), antiviral (remdesivir), and IL-6 receptor blocking monoclonal antibodies (tocilizumab), were used to treat COVID-19 before the vaccines came out.

The paper "Medication use evaluation of tocilizumab implementation in COVID-19 treatment guidelines: A causal inference approach" also provides evidence on the use of medications to treat COVID-19 before the vaccines came out. The study found that the implementation of tocilizumab was associated with a 3.

In [14]:
ast.literal_eval(output.raw)

[{'paperId': '4bdac574e49d69c7d147a111088dc08ac2b534e7',
  'url': 'https://www.semanticscholar.org/paper/4bdac574e49d69c7d147a111088dc08ac2b534e7',
  'title': 'Diabetes medications and associations with Covid-19 outcomes in the N3C database: A national retrospective cohort study',
  'abstract': 'Background While vaccination is the most important way to combat the SARS-CoV-2 pandemic, there may still be a need for early outpatient treatment that is safe, inexpensive, and currently widely available in parts of the world that do not have access to the vaccine. There are in-silico, in-vitro, and in-tissue data suggesting that metformin inhibits the viral life cycle, as well as observational data suggesting that metformin use before infection with SARS-CoV2 is associated with less severe COVID-19. Previous observational analyses from single-center cohorts have been limited by size. Methods Conducted a retrospective cohort analysis in adults with type 2 diabetes (T2DM) for associations betwe

In [3]:
dir(output)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__',
 '__pydantic_serializer__',
 '__pydantic_validator__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__repr_a

In [19]:
[dict(result) for result in results]

KeyboardInterrupt: 

In [14]:
sorted(most_important_fields)

['abstract',
 'authors',
 'citationCount',
 'externalIds',
 'fieldsOfStudy',
 'influentialCitationCount',
 'isOpenAccess',
 'journal',
 'openAccessPdf',
 'paperId',
 'publicationDate',
 'publicationTypes',
 'publicationVenue',
 'referenceCount',
 's2FieldsOfStudy',
 'title',
 'tldr',
 'url',
 'venue',
 'year']

In [9]:
ss = SemanticScholar(retry=True, api_key=os.environ["SEMANTIC_SCHOLAR_API_KEY"])
results = ss.search_paper("CNNs", fields=['abstract', 'tldr'], limit=2)
json.dumps(dict(results.items))

KeyError: 0

In [12]:
import ast
ast.literal_eval(f"{results.items}")

[{'paperId': '0f50b7483f1b200ebf88c4dd7698de986399a0f3',
  'abstract': 'Convolutional Neural Networks (CNNs) are commonly thought to recognise objects by learning increasingly complex representations of object shapes. Some recent studies suggest a more important role of image textures. We here put these conflicting hypotheses to a quantitative test by evaluating CNNs and human observers on images with a texture-shape cue conflict. We show that ImageNet-trained CNNs are strongly biased towards recognising textures rather than shapes, which is in stark contrast to human behavioural evidence and reveals fundamentally different classification strategies. We then demonstrate that the same standard architecture (ResNet-50) that learns a texture-based representation on ImageNet is able to learn a shape-based representation instead when trained on "Stylized-ImageNet", a stylized version of ImageNet. This provides a much better fit for human behavioural performance in our well-controlled psycho

In [9]:
pprint(dict(results[0]))

{'abstract': 'Convolutional Neural Networks (CNNs) are commonly thought to '
             'recognise objects by learning increasingly complex '
             'representations of object shapes. Some recent studies suggest a '
             'more important role of image textures. We here put these '
             'conflicting hypotheses to a quantitative test by evaluating CNNs '
             'and human observers on images with a texture-shape cue conflict. '
             'We show that ImageNet-trained CNNs are strongly biased towards '
             'recognising textures rather than shapes, which is in stark '
             'contrast to human behavioural evidence and reveals fundamentally '
             'different classification strategies. We then demonstrate that '
             'the same standard architecture (ResNet-50) that learns a '
             'texture-based representation on ImageNet is able to learn a '
             'shape-based representation instead when trained on '
            

In [ ]:
def semantic_scholar_query(query: str):
    ss = SemanticScholar(retry=False, api_key=os.environ["SEMANTIC_SCHOLAR_API_KEY"])
    results = ss.search_paper(
        query,
        bulk=False,
        fields=['title', 'tldr', 'abstract'],
    )

In [5]:
import logging
logging.basicConfig(level=logging.DEBUG)
ss = SemanticScholar(retry=False)
results = ss.search_paper(
    "CNNs",
    # fields=['title', 'tldr', 'abstract'],
    # limit=20,
)
results.total

RetryError: RetryError[<Future at 0x22d11fef350 state=finished raised ConnectionRefusedError>]

In [20]:
len(results)

6

In [16]:
from pprint import pprint
for result in results:
    if "tldr" in dict(result):
        print(result.title)
        pprint(result.tldr)
        print()

Towards Broadening the COVID-19 Knowledge Base
None

The COVID-19 Pandemic and Its Impact on Dentistry.
None

Gynecology in the Time of Coronavirus
None

Delivering a difference.
None

The Mirage of Health
None

Science fiction has become reality: Best practice for future viral pandemics
None



In [14]:
"title" in results[0]

KeyError: 0

In [2]:
crew = fakt_ai_crew()
r = crew.kickoff({
    "query": "What treatments were available to treat covid before the vaccines came out?"
})
display(Markdown(r.raw))

 [2024-09-09 15:49:57][DEBUG]: == Working Agent: Fakt AI
 [2024-09-09 15:49:57][INFO]: == Starting Task: Our goal is to determine the truth or falisty of the following query
        submitted by a user:

        Query: What treatments were available to treat covid before the vaccines came out?

        You are Fakt AI, an agent designed to scour the internet and determine whether a given
        user query is True or False. You exist to cut through the noise and combat the mass of
        information online that no single person can possibly handle themselves. It is your goal
        to gather relavant info and synthesis it into comprehensive answers.

        Some use cases include:
        - Alice listens to a podcast and hears Bob say: X is true and you can read scientific
        paper Y for proof. 

            - Fakt AI, will search online for that paper, read the paper, find the relevant portions
            and present back an argument as to whether that is true or not.

      

# COVID-19 Treatments Before Vaccines
## Introduction
The COVID-19 pandemic has led to a significant global effort to develop effective treatments and vaccines. Before the vaccines became available, several treatments were used to manage the disease. This document provides an overview of the treatments available to treat COVID-19 before the vaccines came out.

## Treatments
Several treatments were used to manage COVID-19 before the vaccines became available. These include:

* Remdesivir: A study published in PubMed found that remdesivir therapy was associated with improved survival in patients with severe COVID-19 [1].
* Antiviral medications: A review published in Semantics Scholar found that several antiviral medications, including lopinavir/ritonavir and oseltamivir, were used to treat COVID-19 [2].
* Corticosteroids: A study published in PubMed found that corticosteroids were effective in reducing mortality in patients with severe COVID-19 [3].
* Immunomodulatory therapies: A review published in Semantics Scholar found that several immunomodulatory therapies, including tocilizumab and anakinra, were used to treat COVID-19 [2].

## Vaccines
The development of vaccines was a critical step in controlling the COVID-19 pandemic. Several vaccines were developed and approved for emergency use, including the Pfizer-BioNTech and Moderna vaccines.

## Conclusion
In conclusion, several treatments were available to treat COVID-19 before the vaccines came out. These treatments included remdesivir, antiviral medications, corticosteroids, and immunomodulatory therapies. The development of vaccines was a critical step in controlling the pandemic, and several vaccines were developed and approved for emergency use.

## References
[1] PubMed: SARS-CoV-2 viral load is linked to remdesivir efficacy in severe Covid-19 admitted to intensive care.

[2] Semantics Scholar: COVID-19 Treatments and Vaccines: A year in Review.

[3] PubMed: Corticosteroids for COVID-19: a systematic review and meta-analysis.

Note: The references provided are a selection of the sources used to gather information for this document. They are not an exhaustive list of all sources used.

In [ ]:
from langchain_community.tools.semanticscholar.tool import SemanticScholarQueryRun

tool = SemanticScholarQueryRun()

tool.invoke({"query": "What are some biases in the large language models? How have people tried to mitigate them?"})

In [1]:
from langchain_community.utilities.semanticscholar import SemanticScholarAPIWrapper
ss = SemanticScholarAPIWrapper(
    top_k_results = 3,
    load_max_docs = 3
)
ss.run("biases in large language models")

'Published year: 2023\nTitle: Biases in Large Language Models: Origins, Inventory, and Discussion\nAuthors: Roberto Navigli, Simone Conia, Björn Ross\nAbstract: In this article, we introduce and discuss the pervasive issue of bias in the large language models that are currently at the core of mainstream approaches to Natural Language Processing (NLP). We first introduce data selection bias, that is, the bias caused by the choice of texts that make up a training corpus. Then, we survey the different types of social bias evidenced in the text generated by language models trained on such corpora, ranging from gender to age, from sexual orientation to ethnicity, and from religion to culture. We conclude with directions focused on measuring, reducing, and tackling the aforementioned types of bias.\n\n\nPublished year: 2023\nTitle: Benchmarking Cognitive Biases in Large Language Models as Evaluators\nAuthors: Ryan Koo, Minhwa Lee, Vipul Raheja, Jong Inn Park, Zae Myung Kim, Dongyeop Kang\nAb

In [5]:
display(Markdown(r.raw))

1. **Title:** Finding Effective Treatments for COVID-19: Scientific Integrity and Public Confidence in a Time of Crisis.
   - **Authors:** J. L. Goodman, L. Borio
   - **Published Year:** 2020
   - **Abstract:** Everyone wants new treatments and vaccines to address the devastation of coronavirus disease 2019 (COVID-19). But, currently, under intense pressure and based on hope and limited data from poorly conducted clinical trials and observational data, many clinicians are embarking on ill-advised and uncontrolled human experimentation with unproven treatments. This approach cannot provide answers about what treatments are effective, and it poses undue risk to patients. In this light, decisions to seek and invoke Emergency Use Authorization (EUA) authorities from the US Food and Drug Administration (FDA), such as the recent EUA for chloroquine and hydroxychloroquine, which will further increase use of these drugs for treating individuals with COVID-19, are noteworthy and deserve careful attention. Not only are there potential negative consequences from uncontrolled use of these drugs based on currently unconvincing data but, equally concerning, the integrity of governmental decision-making is increasingly coming under pressure, risking harm to both patients and to the public confidence needed to respond effectively to this pandemic.
   - **Reason:** This paper highlights the use of chloroquine and hydroxychloroquine under EUA for treating COVID-19, despite the limited data supporting their effectiveness.

2. **Title:** Current pharmacological treatments for COVID‐19: What's next?
   - **Authors:** C. Scavone, Simona Brusco, M. Bertini, Liberata Sportiello, C. Rafaniello, A. Zoccoli, L. Berrino, G. Racagni, F. Rossi, A. Capuano
   - **Published Year:** 2020
   - **Abstract:** Since December 2019 SARS‐Cov‐2 was found responsible for the disease COVID‐19, which has spread worldwide. No specific therapies/vaccines are yet available for the treatment of COVID‐19. Drug repositioning may offer a strategy and a number of drugs have been repurposed, including lopinavir/ritonavir, remdesivir, favipiravir and tocilizumab. This paper describes the main pharmacological properties of such drugs administered to patients with COVID‐19, focusing on their antiviral, immune‐modulatory and/or anti‐inflammatory actions. Where available, data from clinical trials involving patients with COVID‐19 are reported. Preliminary clinical trials seem to support their benefit. However, such drugs in COVID‐19 patients have peculiar safety profiles. Thus, adequate clinical trials are necessary for these compounds. Nevertheless, while waiting for effective preventive measures i.e. vaccines, many clinical trials on drugs belonging to different therapeutic classes are currently underway. Their results will help us in defining the best way to treat COVID‐19 and reducing its symptoms and complications.
   - **Reason:** This paper discusses several repurposed drugs such as lopinavir/ritonavir, remdesivir, favipiravir, and tocilizumab, which were considered for treating COVID-19 before the availability of vaccines.

3. **Title:** The production of mRNA vaccines in the COVID-19 pandemic: advent of new applications in health treatments
   - **Authors:** Ana Luiza Barros da Silva, A. D. de Araújo, Vitor Hugo Auzier Lima, Ciana Martins De Oliveira
   - **Published Year:** 2023
   - **Abstract:** Modern vaccines developed using mRNA technology are highly accurate formulations designed to stimulate the production of a variety of antigenic properties in the user in order to activate adaptive immunity. This enables the use of such technology in the research of vaccines against several infectious diseases, such as hepatitis C, Acquired Immunodeficiency Syndrome (AIDS), malaria, tuberculosis, in addition to chronic diseases, such as some types of cancer. Such vaccines were an efficient alternative in the fight against COVID-19 and have remarkable characteristics, such as the versatility of their applications and the ability to encode and express virtually any type of protein. Like this, this work is justified by the need to know the potential applications of this technology in clinical practice and in the search for effective alternative solutions.
   - **Reason:** This paper provides context on the development of mRNA vaccines and their role in fighting COVID-19, but it is less relevant to the query about treatments before vaccines were available.

These papers provide a comprehensive look at the treatments that were considered and used for COVID-19 before vaccines were released, focusing on repurposed drugs and emergency use authorizations.